# Template Tuning with DimRed API

This notebook demonstrates how to run prompt tuning with template variables using the DimRed API.

## Overview

The workflow:
1. Load template dataset from `template_optimization_example.json`
2. Create a project and dataset
3. Add datapoints with template_var payloads
4. Create a prompt with template variable placeholders (e.g., `{{customer_name}}`)
5. Create a metric
6. Run tuning
7. Poll for results

## Setup

In [ ]:
import json
import logging
import sys
import os
from pathlib import Path

# Add the parent directory (dimred-examples) to Python path for importing client
# This handles both running from notebooks and via nbconvert
current_dir = os.getcwd()
dimred_root = current_dir

# Find the dimred-examples directory
if 'dimred-examples' in current_dir:
    # Split path and find dimred-examples root
    parts = current_dir.split('/')
    idx = next(i for i, p in enumerate(parts) if 'dimred-examples' in p)
    dimred_root = '/'.join(parts[:idx+1])

if dimred_root not in sys.path:
    sys.path.insert(0, dimred_root)

from client import DimRedAPIClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
# API Configuration
API_KEY = os.environ.get("DIMRED_API_KEY")
BASE_URL = "https://www.dimred.com"  # Updated URL

# Path to template dataset
current_dir = os.getcwd()
dimred_root = current_dir

# Find the dimred-examples directory
if 'dimred-examples' in current_dir:
    # Split path and find dimred-examples root
    parts = current_dir.split('/')
    idx = next(i for i, p in enumerate(parts) if 'dimred-examples' in p)
    dimred_root = '/'.join(parts[:idx+1])

TEMPLATE_DATASET_PATH = os.path.join(dimred_root, 'data', 'template_optimization_example.json')

# Initialize client
client = DimRedAPIClient(API_KEY, BASE_URL)
print(f"✓ Initialized DimRed API client")
print(f"✓ Dataset path: {TEMPLATE_DATASET_PATH}")

## Step 1: Load Template Dataset

The dataset contains datapoints with template variable payloads. Each datapoint has:
- `input`: The article snippet to analyze
- `expected`: The expected response (is_perpetrator and reasoning)
- `payloads`: Array of template_var payloads with key/value pairs for substitution

In [13]:
# Load the template dataset
with open(TEMPLATE_DATASET_PATH, 'r') as f:
    template_dataset = json.load(f)

print(f"Loaded {len(template_dataset)} datapoints from {TEMPLATE_DATASET_PATH}")
print(f"\nFirst datapoint structure:")
print(f"  - input keys: {list(template_dataset[0]['input'].keys())}")
print(f"  - expected keys: {list(template_dataset[0]['expected'].keys())}")
print(f"  - Number of payloads: {len(template_dataset[0]['payloads'])}")
print(f"  - Payload type: {template_dataset[0]['payloads'][0]['payload_type']}")
print(f"  - Template variable: {template_dataset[0]['payloads'][0]['payload']['key']} = {template_dataset[0]['payloads'][0]['payload']['value']}")

Loaded 12 datapoints from /Users/jonathanjohannemann/dimred-examples/data/template_optimization_example.json

First datapoint structure:
  - input keys: ['article_snippet']
  - expected keys: ['is_perpetrator', 'reasoning']
  - Number of payloads: 1
  - Payload type: template_var
  - Template variable: customer_name = Carlos Mendes


## Step 2: Create Project

In [14]:
project_id = client.create_project(
    project_name="Perpetrator Classification Tuning",
    project_description="Testing template-based prompt tuning for financial crime perpetrator identification"
)

print(f"✓ Created project: {project_id}")

[2025-10-27 20:57:51] INFO - Creating project: Perpetrator Classification Tuning
[2025-10-27 20:57:52] INFO - ✓ Project created: dcac150f-e43d-4f2e-9ea3-364ecadc582e
✓ Created project: dcac150f-e43d-4f2e-9ea3-364ecadc582e


## Step 3: Create Dataset

In [15]:
dataset_id = client.create_dataset(
    project_id=project_id,
    dataset_name="Perpetrator Classification Dataset"
)

print(f"✓ Created dataset: {dataset_id}")

[2025-10-27 20:57:52] INFO - Creating dataset: Perpetrator Classification Dataset for project dcac150f-e43d-4f2e-9ea3-364ecadc582e
[2025-10-27 20:57:53] INFO - ✓ Dataset created: ds-6292bdb6-a2ee-4ac0-b923-e76f3502a8cd
✓ Created dataset: ds-6292bdb6-a2ee-4ac0-b923-e76f3502a8cd


## Step 4: Add Datapoints with Template Variable Payloads

When adding datapoints with template variables, the API expects:
- `input_data`: JSON string of the input
- `expected_output`: JSON string of the expected output
- `payloads`: Array of payload objects with structure:
  ```json
  {
    "payload_type": "template_var",
    "payload": {
      "key": "customer_name",
      "value": "John Doe"
    }
  }
  ```

The system will replace `{{customer_name}}` in the prompt with the provided value.

In [16]:
# Convert dataset to API format
# Rename 'input' -> 'input_data' and 'expected' -> 'expected_output'
datapoints = []
for item in template_dataset:
    datapoint = {
        "input_data": json.dumps(item["input"]),
        "expected_output": json.dumps(item["expected"]),
        "payloads": item["payloads"]  # Pass payloads as-is
    }
    datapoints.append(datapoint)

# Add to dataset
count = client.add_datapoints(dataset_id, datapoints)
print(f"✓ Added {count} datapoints with template_var payloads")

# Display an example datapoint structure
print("\n=== Example Datapoint Structure ===")
example = datapoints[0]
print(f"\ninput_data: {example['input_data']}")
print(f"\nexpected_output: {example['expected_output']}")
print(f"\npayloads: {len(example['payloads'])} payload(s)")
print(f"\nPayload details:")
for i, payload in enumerate(example['payloads'], 1):
    print(f"  Payload {i}:")
    print(f"    - Type: {payload['payload_type']}")
    print(f"    - Template var: {payload['payload']['key']} = {payload['payload']['value']}")

[2025-10-27 20:57:53] INFO - Adding 12 datapoints to dataset ds-6292bdb6-a2ee-4ac0-b923-e76f3502a8cd
[2025-10-27 20:57:54] INFO - ✓ Added 12 datapoints
✓ Added 12 datapoints with template_var payloads

=== Example Datapoint Structure ===

input_data: {"article_snippet": "Federal agents announced the arrest of Carlos Mendes, 47, accused of funneling over $5 million in drug proceeds through a chain of shell corporations across three states. His business partner Maria Santos, 43, was also questioned but released without charges after cooperating with investigators. Court documents reveal that Mendes disguised cash deposits as legitimate sales revenue before transferring funds overseas. The investigation began after suspicious activity reports filed by his bank's compliance department."}

expected_output: {"is_perpetrator": true, "reasoning": "Carlos Mendes was formally arrested and charged with money laundering through shell corporations, while his partner Maria Santos was released withou

## Step 5: Create Prompt

Create a prompt for perpetrator classification with template variable placeholders.
The `{{customer_name}}` placeholder will be replaced with the actual value from the payload.

In [ ]:
# Create prompt text with template variable placeholder
prompt_text = (
    "You are an expert analyst evaluating whether individuals are perpetrators in financial crime cases. "
    "Your task is to analyze an article snippet and determine if the person named '{{customer_name}}' "
    "is a perpetrator based on the evidence provided.\n\n"
    "Guidelines:\n"
    "- Consider formal charges, arrests, and documented evidence\n"
    "- Distinguish between perpetrators and cooperating witnesses\n"
    "- Account for presumption of innocence when investigations are ongoing\n"
    "- Look for concrete evidence vs. circumstantial patterns\n\n"
    "Respond with JSON containing:\n"
    "- is_perpetrator: true or false\n"
    "- reasoning: detailed explanation of your decision based on the evidence"
)

# Output schema for structured response
output_schema = {
    "type": "object",
    "properties": {
        "is_perpetrator": {
            "type": "boolean",
            "description": "Whether the specified customer_name is determined to be a perpetrator"
        },
        "reasoning": {
            "type": "string",
            "description": "Detailed explanation of the decision based on available evidence"
        }
    },
    "required": ["is_perpetrator", "reasoning"],
    "additionalProperties": False
}

# Use new API format
prompt_id = client.create_prompt(
    project_id=project_id,
    prompt_text=prompt_text,
    prompt_message_type="system",
    name="Perpetrator Classification with Template Variables",
    output_schema=output_schema
)

print(f"✓ Created prompt: {prompt_id}")
print(f"\nPrompt uses template variable: {{{{customer_name}}}}")

## Step 6: Create Metric

In [18]:
metric_code = '''
import json

def metric_func(output, expected):
    """
    Check if the model correctly identified whether the person is a perpetrator.
    Returns 1.0 for correct classification, 0.0 for incorrect.
    """
    # Parse output and expected if they're strings
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return 0.0

    if isinstance(expected, str):
        try:
            expected = json.loads(expected)
        except json.JSONDecodeError:
            return 0.0

    # Extract is_perpetrator field
    output_value = output.get("is_perpetrator")
    expected_value = expected.get("is_perpetrator")

    # Both must be present and match
    if output_value is None or expected_value is None:
        return 0.0

    # Return 1.0 if they match, 0.0 if they don't
    return 1.0 if output_value == expected_value else 0.0
'''

metric_id = client.create_metric(
    project_id=project_id,
    code=metric_code,
    metric_name="Perpetrator Classification Accuracy",
    metric_description="Measures whether the model correctly identifies perpetrators based on article evidence"
)

print(f"✓ Created metric: {metric_id}")

[2025-10-27 20:57:55] INFO - Creating metric for project dcac150f-e43d-4f2e-9ea3-364ecadc582e
[2025-10-27 20:57:55] INFO - ✓ Metric created: 71a5a09b-2443-42ec-a728-be5809afa050
✓ Created metric: 71a5a09b-2443-42ec-a728-be5809afa050


## Step 7: Run Tuning

Start the prompt tuning session. The model will receive prompts with template variables substituted.

In [ ]:
# Run tuning using the new workflow API
workflow_response = client.run_workflow(
    project_id=project_id,
    dataset_id=dataset_id,
    prompt_id=prompt_id,
    metric_id=metric_id,
    model_name="gpt-4o-mini",  # Updated model name
    provider="openai",
    mode="tune",
    num_iterations=1
)

workflow_id = workflow_response.get("id") or workflow_response.get("tuning_session_id")

print(f"✓ Tuning started")
print(f"  Workflow ID: {workflow_id}")
print(f"  Status: {workflow_response.get('status')}")

## Step 8: Wait for Completion

Poll the tuning session until it completes. This can take 10-30 minutes.

In [ ]:
# Wait for workflow completion using new monitoring
final_result = client.wait_for_workflow_completion(
    workflow_id=workflow_id,
    poll_interval=10,
    timeout=3600
)

print("\n=== Final Results ===")
print(f"Workflow ID: {workflow_id}")
print(f"Status: {final_result.get('status')}")

# Extract metrics data
metrics_data = final_result.get("metrics", {})
final_metrics = metrics_data.get("final_metrics", {})
iteration_results = metrics_data.get("iteration_results", [])

if final_metrics:
    print("\nFinal Metrics:")
    for metric_name, value in final_metrics.items():
        print(f"  {metric_name}: {value}")

if iteration_results:
    print(f"\nCompleted {len(iteration_results)} iterations:")
    for result in iteration_results:
        iter_num = result.get("iteration", "?")
        iter_metrics = result.get("metrics", {})
        print(f"  Iteration {iter_num}: {iter_metrics}")

print(f"\nBest Prompt ID: {metrics_data.get('best_prompt_id', 'N/A')}")
print(f"Final Prompt ID: {metrics_data.get('final_prompt_id', 'N/A')}")

## Step 9: Fetch Best Prompt

## Step 9: Fetch Best Prompt

Retrieve the best performing prompt from the tuning session, which will show the optimized template with placeholders.

In [ ]:
# Fetch and display the best template-based prompt
best_prompt_id = metrics_data.get('best_prompt_id')
final_prompt_id = metrics_data.get('final_prompt_id')

print("\n=== Best Template-Based Prompt ===")

if best_prompt_id:
    print(f"Best Prompt ID: {best_prompt_id}")
    
    if final_prompt_id and final_prompt_id != best_prompt_id:
        print(f"Final Prompt ID: {final_prompt_id}")
    
    # Always fetch and display the best prompt
    print(f"\nFetching prompt details for ID: {best_prompt_id}")
    try:
        best_prompt = client.get_prompt(best_prompt_id)
        
        # Check if we got valid prompt data
        if best_prompt and isinstance(best_prompt, dict):
            if best_prompt.get('prompt_text'):
                print("\n✓ Successfully retrieved prompt")
                print(f"\nTemplate Prompt Text:")
                print("-" * 50)
                print(best_prompt['prompt_text'])
                print("-" * 50)
                
                # Check if template variable was preserved
                if '{{customer_name}}' in best_prompt.get('prompt_text', ''):
                    print("\n✓ Template variable {{customer_name}} preserved in prompt")
                
                if best_prompt.get('output_schema'):
                    print(f"\nOutput Schema:")
                    print(json.dumps(best_prompt['output_schema'], indent=2))
                    
                if best_prompt.get('prompt_message_type'):
                    print(f"\nMessage Type: {best_prompt['prompt_message_type']}")
            else:
                # Empty response, fetch the original prompt as fallback
                print(f"API response empty, fetching original prompt ID: {prompt_id}")
                original_prompt = client.get_prompt(prompt_id)
                if original_prompt and original_prompt.get('prompt_text'):
                    print("\n✓ Retrieved original prompt")
                    print(f"\nTemplate Prompt Text:")
                    print("-" * 50)
                    print(original_prompt['prompt_text'])
                    print("-" * 50)
                    
                    if '{{customer_name}}' in original_prompt.get('prompt_text', ''):
                        print("\n✓ Template variable {{customer_name}} present in prompt")
                else:
                    print("✓ Prompt IDs captured successfully")
        else:
            print("✓ Prompt IDs captured successfully")
            
    except Exception as e:
        print(f"Could not fetch full prompt details: {e}")
        print("✓ Prompt IDs captured successfully")
    
    # Indicate whether optimization occurred
    if prompt_id == best_prompt_id:
        print("\n📊 Result: The original template prompt performed best")
    else:
        print("\n📊 Result: An optimized template prompt was generated")
    
    print("\n📝 Use this prompt ID with template variables in future workflows:")
    print(f"   client.run_workflow(..., prompt_id='{best_prompt_id}', ...)")
    print("   Remember to provide template_variables when running inference!")
    
else:
    print("Error: Best prompt ID not available in results")
    print("Check that the tuning workflow completed successfully.")

## Summary

You've successfully completed template-based prompt tuning with the DimRed API:

- ✓ Loaded template dataset with template_var payloads
- ✓ Created project and dataset
- ✓ Added datapoints with template variable payloads
- ✓ Created prompt with `{{customer_name}}` placeholder
- ✓ Created classification metric
- ✓ Ran tuning with template variable substitution
- ✓ Retrieved optimized prompt

## Key Points for Template Variable Payloads

1. **Template Payload Structure**: Use `payload_type: "template_var"` with `{key, value}` format
2. **Placeholder Syntax**: Use double curly braces in prompts: `{{variable_name}}`
3. **Substitution**: The system automatically replaces placeholders with payload values
4. **Model Selection**: Any text model works (no special capabilities required)
5. **Multiple Variables**: You can have multiple template_var payloads per datapoint
6. **Use Case**: Ideal for testing the same prompt template with different variable values